In [29]:
from graph.builder import format_messages, get_last_human_message
from functools import lru_cache
from typing import Any, Coroutine

from langchain_core.language_models import BaseChatModel
import json
from datetime import datetime
from functools import lru_cache
from pathlib import Path
from uuid import uuid4

from langchain.chat_models import init_chat_model
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langgraph.graph import END, START, StateGraph
from pydantic import AliasChoices, BaseModel, Field

from config.logging import trace_event
from config.settings import settings
from graph.state import AgentState
from memory.postgres_log import save_conversation_log

class IntentResult(BaseModel):
    current_intent: str = Field(
        description="用户当前意图",
        validation_alias=AliasChoices("current_intent", "intent"),
    )
    current_order_type: str | None = Field(
        default=None,
        description="订单类型",
        validation_alias=AliasChoices("current_order_type", "order_type"),
    )

@lru_cache
def get_llm() -> BaseChatModel:
    return init_chat_model(
        model=settings.openai_model,
        model_provider="openai",
        base_url=settings.openai_base_url,
        api_key=settings.openai_api_key,
        temperature=settings.openai_temperature,
    )

# 安全地获取文件路径，兼容 Jupyter Notebook 等环境
try:
    PROMPTS_DIR = Path(__file__).resolve().parents[1] / "prompts"
except NameError:
    # 当 __file__ 未定义时（如在 Jupyter 中），使用当前工作目录
    PROMPTS_DIR = Path.cwd() / "prompts"

print(PROMPTS_DIR)

def to_prompt_text(value: object) -> str:
    if isinstance(value, str):
        return value
    return json.dumps(value, ensure_ascii=False, default=str)

def load_prompt(relative_path: str) -> str:
    return (PROMPTS_DIR / relative_path).read_text(encoding="utf-8")

def render_prompt(relative_path: str, **variables: object) -> str:
    prompt = load_prompt(relative_path)
    for key, value in variables.items():
        prompt = prompt.replace(f"{{{{{key}}}}}", to_prompt_text(value))
    return prompt

async def intent_node(state: AgentState) -> Any:
    """识别用户当前维修意图和订单类型。"""

    llm = get_llm()
    result = await llm.ainvoke(
        [
            SystemMessage(
                content=render_prompt(
                    "router/maintenance_intent_router.md",
                    conversation_history=format_messages(state["messages"]),
                    last_user_message=get_last_human_message(state["messages"]),
                )
            ),
        ]
    )

    print("result begin")


    print("result end")


    return result


print(render_prompt(
    "router/maintenance_intent_router.md",
))





/Users/chengzongxin/project/hotel-ai-order-cursor/prompts
你是酒店 AI 维修下单系统的意图识别器。

请输出 JSON object，且只能输出符合 schema 的 JSON。

可选意图：
- create_repair_order：用户要创建维修单、报修、反馈设备故障。
- confirm_repair_order：用户确认提交维修单。
- cancel_repair_order：用户取消维修单。
- smalltalk：普通闲聊。
- unknown：无法判断。

判断规则：
1. 判断时优先看“用户最新输入”，不要被较早历史中的维修内容误导。
2. 只要用户最新输入提到维修、报修、设备坏了、漏水、不亮、不制冷、打不开、堵塞等，都属于 create_repair_order。
3. 如果用户最新输入是在确认提交，例如“确认”“提交”“没问题”“就这样”，属于 confirm_repair_order。
4. 如果用户最新输入是闲聊、问时间、问天气、试探系统能力、无关内容，属于 smalltalk。
5. 如果是维修相关请求，current_order_type 必须返回 repair_order。
6. 如果只是闲聊或完全无关，current_order_type 返回 null。
7. 不要输出 Markdown。
8. 不要输出解释。

对话历史：
{{conversation_history}}

用户最新输入：
{{last_user_message}}



In [28]:

# 测试不同的用户输入
test_cases = [
    "你好，我房间的空调坏了",
    "我要点餐",
    "帮我取消订单",
]
for user_input in test_cases:
    print(f"\n测试输入: {user_input}")
    test_state: AgentState = {
        "messages": [HumanMessage(content=user_input)],
        "conversation_id": str(uuid4()),
        "last_user_message": user_input,
    }

    result = await intent_node(test_state)
    print(f"结果: {result}\n")


测试输入: 你好，我房间的空调坏了
result begin
result end
结果: content='{"intent": "create_repair_order", "current_order_type": "repair_order"}' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 1005, 'prompt_tokens': 303, 'total_tokens': 1308, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 982, 'rejected_prediction_tokens': None, 'text_tokens': 1005}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 303}}, 'model_provider': 'openai', 'model_name': 'qwen3.5-plus', 'system_fingerprint': None, 'id': 'chatcmpl-d3201deb-21da-9fb6-a77a-86c6ad20e364', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e4460-abd4-78d3-830c-ecf0d743bb33-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 303, 'output_tokens': 1005, 'total_tokens': 1308, 'input_token_details': {}, 'output_token_details': {'reasoning': 982}}


测试输入: 我要点餐
result begin
result end
结果: